In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Point this at your run folder ─────────────────────────────────────────────
RUN_FOLDER = Path("data/multi_run/output/20260408_140829_scenario_400")
# ─────────────────────────────────────────────────────────────────────────────

npz  = np.load(RUN_FOLDER / "timeseries.npz")
keys = sorted(npz.files)

# Time axis: DataCollector records every 10 min starting at t=10
N_STEPS    = npz[keys[0]].shape[1]
time_axis  = np.arange(1, N_STEPS + 1) * 10   # minutes
time_hours = time_axis / 60

# Group keys by metric and facility
transport_facs = sorted({k.split('_T')[0] for k in keys if k.startswith('ts_transport')})
dcs_facs       = sorted({k.split('_T')[0] for k in keys if k.startswith('ts_dcs')})

print(f"Runs: {npz[keys[0]].shape[0]}")
print(f"Transport facilities: {[f.replace('ts_transport_','') for f in transport_facs]}")
print(f"DCS facilities:       {[f.replace('ts_dcs_','') for f in dcs_facs]}")

def plot_triage(triage, ax=None):
    """Plot mean queue length ± IQR for all facilities for one triage level."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 4))

    colors_transport = plt.cm.Blues(np.linspace(0.4, 0.9, len(transport_facs)))
    colors_dcs       = plt.cm.Oranges(np.linspace(0.4, 0.9, len(dcs_facs)))

    for fac, color in zip(transport_facs, colors_transport):
        key  = f"{fac}_{triage}"
        if key not in npz.files:
            continue
        data = npz[key]                        # (n_runs, n_steps)
        mean = data.mean(axis=0)
        p25  = np.percentile(data, 25, axis=0)
        p75  = np.percentile(data, 75, axis=0)
        label = fac.replace('ts_transport_', 'R1 ') + ' (transport)'
        ax.plot(time_hours, mean, color=color, linewidth=1.5, label=label)
        ax.fill_between(time_hours, p25, p75, color=color, alpha=0.15)

    for fac, color in zip(dcs_facs, colors_dcs):
        key  = f"{fac}_{triage}"
        if key not in npz.files:
            continue
        data = npz[key]
        mean = data.mean(axis=0)
        p25  = np.percentile(data, 25, axis=0)
        p75  = np.percentile(data, 75, axis=0)
        label = fac.replace('ts_dcs_', 'R2 ') + ' (DCS queue)'
        ax.plot(time_hours, mean, color=color, linewidth=1.5, linestyle='--', label=label)
        ax.fill_between(time_hours, p25, p75, color=color, alpha=0.15)

    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Patients waiting')
    ax.set_title(f'{triage} — queue length over time (mean ± IQR across runs)')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    return ax

In [ ]:
# ── T1 ────────────────────────────────────────────────────────────────────────
plot_triage('T1')
plt.tight_layout()
plt.show()

In [ ]:
# ── T2 ────────────────────────────────────────────────────────────────────────
plot_triage('T2')
plt.tight_layout()
plt.show()

In [ ]:
# ── T3 ────────────────────────────────────────────────────────────────────────
plot_triage('T3')
plt.tight_layout()
plt.show()